# Aviation Accidents Analysis
This notebook represents the fully optimized **Phase 1 to Phase 3** analysis, ensuring robust descriptive statistics, error-free plotting, business-driven factor analysis, and automatic image exporting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Create images directory for exported visuals
os.makedirs('images', exist_ok=True)

## Exploratory Data Analysis
Load in the cleaned data.

In [ ]:
df = pd.read_csv('data/aviation_cleaned.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## Explore safety metrics across makes
We calculate the accident count, the median, and the mean to give a full statistical picture.

In [ ]:
if not df.empty:
    # Separate by passenger threshold
    small_planes = df[df['occupants_est'] <= 20].copy()
    large_planes = df[df['occupants_est'] > 20].copy()
    
    # 1. Robust Statistics for Small Makes
    small_make_stats = small_planes.groupby('Make_clean').agg(
        accident_count=('Event.Id', 'count'),
        injury_mean=('fatal_serious_fraction', 'mean'),
        injury_median=('fatal_serious_fraction', 'median'),
        dest_mean=('destroyed', 'mean')
    ).reset_index()
    
    # 2. Robust Statistics for Large Makes
    large_make_stats = large_planes.groupby('Make_clean').agg(
        accident_count=('Event.Id', 'count'),
        injury_mean=('fatal_serious_fraction', 'mean'),
        injury_median=('fatal_serious_fraction', 'median'),
        dest_mean=('destroyed', 'mean')
    ).reset_index()

    # Sort to find the safest 10 based on lowest injury mean, then lowest destruction mean
    top_small_makes = small_make_stats.sort_values(by=['injury_mean', 'dest_mean']).head(10)
    top_large_makes = large_make_stats.sort_values(by=['injury_mean', 'dest_mean']).head(10)

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Small Makes Plot
    sns.barplot(
        data=top_small_makes, x='injury_mean', y='Make_clean', 
        hue='Make_clean', palette='Blues_r', legend=False, ax=axes[0]
    )
    axes[0].set_title('Top 10 Safest Small Makes (Injury Rate)')
    axes[0].set_xlabel('Mean Fatal/Serious Injury Fraction')
    axes[0].set_ylabel('Make')

    # Large Makes Plot
    sns.barplot(
        data=top_large_makes, x='injury_mean', y='Make_clean', 
        hue='Make_clean', palette='Oranges_r', legend=False, ax=axes[1]
    )
    axes[1].set_title('Top 10 Safest Large Makes (Injury Rate)')
    axes[1].set_xlabel('Mean Fatal/Serious Injury Fraction')
    axes[1].set_ylabel('')

    plt.tight_layout()
    plt.savefig('images/makes_injury_rates.png', bbox_inches='tight')
    plt.show()

### Distributions for Makes
Plotting distributions is vital because the data is highly zero-inflated.

In [ ]:
if not df.empty:
    # Small Makes Violin Plot
    small_dist_data = small_planes[small_planes['Make_clean'].isin(top_small_makes['Make_clean'])]
    plt.figure(figsize=(12, 6))
    sns.violinplot(
        data=small_dist_data, x='fatal_serious_fraction', y='Make_clean', 
        hue='Make_clean', order=top_small_makes['Make_clean'], 
        inner="quartile", palette='Blues_r', legend=False
    )
    plt.title('Distribution of Injury Rates: Safest Small Makes')
    plt.xlabel('Fatal/Serious Injury Fraction')
    plt.ylabel('Make')
    plt.savefig('images/small_makes_violin.png', bbox_inches='tight')
    plt.show()

    # Large Makes Strip Plot
    large_dist_data = large_planes[large_planes['Make_clean'].isin(top_large_makes['Make_clean'])]
    plt.figure(figsize=(12, 6))
    sns.stripplot(
        data=large_dist_data, x='fatal_serious_fraction', y='Make_clean', 
        hue='Make_clean', order=top_large_makes['Make_clean'], 
        palette='Oranges_r', alpha=0.6, jitter=True, legend=False
    )
    plt.title('Distribution of Injury Rates: Safest Large Makes')
    plt.xlabel('Fatal/Serious Injury Fraction')
    plt.ylabel('Make')
    plt.savefig('images/large_makes_strip.png', bbox_inches='tight')
    plt.show()

## Analyze plane types (Models)
We must enforce a minimum sample size to avoid skewed data from models with only 1 or 2 historical accidents.

In [ ]:
if not df.empty:
    # 1. Aggregate and enforce minimum sample size of 20 accidents for Models
    small_model_stats = small_planes.groupby('make_model').agg(
        accident_count=('Event.Id', 'count'),
        injury_mean=('fatal_serious_fraction', 'mean'),
        dest_mean=('destroyed', 'mean')
    ).reset_index()
    
    # FILTER: Only keep models with at least 20 historical events
    small_model_stats = small_model_stats[small_model_stats['accident_count'] >= 20]
    top_small_models = small_model_stats.sort_values(by=['injury_mean', 'dest_mean']).head(10)

    # 2. Plotting the top 10 Small Models
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=top_small_models, x='injury_mean', y='make_model', 
        hue='make_model', palette='Blues_d', legend=False
    )
    plt.title('Top 10 Safest Small Aircraft Models (Min. 20 Events)')
    plt.xlabel('Mean Fatal/Serious Injury Fraction')
    plt.ylabel('Aircraft Model')
    plt.savefig('images/small_models_injury.png', bbox_inches='tight')
    plt.show()

In [ ]:
if not df.empty:
    # 1. Aggregate for Large Models (min 10 events to ensure enough candidates for large models)
    large_model_stats = large_planes.groupby('make_model').agg(
        accident_count=('Event.Id', 'count'),
        injury_mean=('fatal_serious_fraction', 'mean'),
        dest_mean=('destroyed', 'mean')
    ).reset_index()
    
    large_model_stats = large_model_stats[large_model_stats['accident_count'] >= 10]
    top_large_models = large_model_stats.sort_values(by=['injury_mean', 'dest_mean']).head(10)

    # 2. Plotting the top 10 Large Models
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=top_large_models, x='injury_mean', y='make_model', 
        hue='make_model', palette='Oranges_d', legend=False
    )
    plt.title('Top 10 Safest Large Aircraft Models (Min. 10 Events)')
    plt.xlabel('Mean Fatal/Serious Injury Fraction')
    plt.ylabel('Aircraft Model')
    plt.savefig('images/large_models_injury.png', bbox_inches='tight')
    plt.show()

### Discussion of Specific Airplane Types
- **Small Aircraft**: Cessna and Piper variants (e.g. Cessna 172, Piper PA-28) have high historical accident volume but extremely low fatal/serious injury rates, making them incredibly robust candidates.
- **Large Aircraft**: Boeing (e.g. 737) and McDonnell Douglas show excellent safety profiles in terms of passenger injury severity.

## Exploring Other Variables
We investigate how Engine Type and Weather Condition impact the safety profile.

In [ ]:
if not df.empty:
    # Factor 1: Weather Condition (VMC vs IMC)
    weather_stats = df[df['Weather.Condition'].isin(['VMC', 'IMC'])].groupby('Weather.Condition').agg(
        accident_count=('Event.Id', 'count'),
        injury_mean=('fatal_serious_fraction', 'mean'),
        dest_mean=('destroyed', 'mean')
    ).reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.barplot(data=weather_stats, x='Weather.Condition', y='injury_mean', hue='Weather.Condition', palette='coolwarm', ax=axes[0], legend=False)
    axes[0].set_title('Mean Injury Rate by Weather Condition')
    axes[0].set_ylabel('Fatal/Serious Fraction')
    
    sns.barplot(data=weather_stats, x='Weather.Condition', y='dest_mean', hue='Weather.Condition', palette='coolwarm', ax=axes[1], legend=False)
    axes[1].set_title('Destruction Rate by Weather Condition')
    axes[1].set_ylabel('Fraction of Aircraft Destroyed')
    
    plt.tight_layout()
    plt.savefig('images/weather_factor.png', bbox_inches='tight')
    plt.show()

In [ ]:
if not df.empty:
    # Factor 2: Engine Type
    engine_data = df[df['Engine.Type'] != 'UNKNOWN']
    
    engine_stats = engine_data.groupby('Engine.Type').agg(
        accident_count=('Event.Id', 'count'),
        injury_mean=('fatal_serious_fraction', 'mean')
    ).reset_index()
    
    # Filter out extremely rare engine types for a robust comparison (min 50 events)
    engine_stats = engine_stats[engine_stats['accident_count'] >= 50].sort_values('injury_mean')
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=engine_stats, x='injury_mean', y='Engine.Type', hue='Engine.Type', palette='viridis', legend=False)
    plt.title('Mean Injury Rate by Engine Type')
    plt.xlabel('Fatal/Serious Fraction')
    plt.ylabel('Engine Type')
    plt.savefig('images/engine_factor.png', bbox_inches='tight')
    plt.show()

### Business Recommendation on Factors
- **Weather (IMC vs VMC)**: Instrument Meteorological Conditions (IMC) significantly increase both the rate of total aircraft destruction and the severity of passenger injuries compared to Visual Meteorological Conditions (VMC). Insurers should mandate strict IFR (Instrument Flight Rules) training and updated avionics for policies covering flight in IMC.
- **Engine Type**: Turbo Fan and Turbo Jet engines consistently show lower severe injury rates compared to Reciprocating engines. This is partly due to the fact that they are typically equipped on larger, more rigidly built commercial aircraft operated by professional crews under stricter regulations.